# Laboratório da Gold

O planejamento da Gold deixou três decisões para resolver com o dado na mão, antes de escrever o script de produção:

- **Decisão 1** — a taxa oficial usa média ponderada pelo `peso_aluno` ou contagem simples?
- **Decisão 4** — qual é o denominador do indicador (presentes com nota, presentes, todos)?
- **Decisão 5** — qual snapshot de meta vale, já que as metas têm linhas por ano de referência e houve revisão entre snapshots?

A régua é a mesma das outras camadas: comparar cada alternativa com o gabarito oficial (`silver/resultados/`) e só levar para o script o que bater.

In [1]:
from pathlib import Path

import pandas as pd

SILVER = Path("../data/silver")
pd.set_option("display.max_columns", 40)

alunos = pd.read_parquet(SILVER / "alunos",
                         columns=["ano", "id_municipio", "rede_nome", "proficiencia",
                                  "peso_aluno", "presente", "sem_nota"])
municipio = pd.read_parquet(SILVER / "resultados" / "municipio")
metas = pd.read_parquet(SILVER / "metas")

print(f"alunos: {len(alunos):,} | resultados municipio: {len(municipio):,} | metas: {len(metas):,}")

alunos: 3,867,589 | resultados municipio: 23,995 | metas: 10,788


## Decisão 1 — ponderada ou simples?

Recalculo a taxa de alfabetização por (ano, município) das duas formas — média simples e média ponderada pelo `peso_aluno` — sempre sobre presentes com nota, e comparo com a `taxa_alfabetizacao` oficial.

Um detalhe que importa: o gabarito municipal é **por rede** (`municipal`, `estadual`, `publica`, `total`), então o confronto honesto é rede a rede. A rede `publica` do gabarito equivale a municipal + estadual dos microdados; a `total` inclui tudo (até a privada residual).

In [2]:
com_nota = alunos[alunos["presente"] & ~alunos["sem_nota"]].copy()
com_nota["alfa"] = (com_nota["proficiencia"] >= 743).astype(float)
com_nota["alfa_peso"] = com_nota["alfa"] * com_nota["peso_aluno"]


def compara_com_gabarito(sub, rede_gabarito):
    g = sub.groupby(["ano", "id_municipio"]).agg(
        soma_alfa=("alfa", "sum"), n=("alfa", "size"),
        soma_alfa_peso=("alfa_peso", "sum"), soma_peso=("peso_aluno", "sum"),
    ).reset_index()
    g["simples"] = 100 * g["soma_alfa"] / g["n"]
    g["ponderada"] = 100 * g["soma_alfa_peso"] / g["soma_peso"]
    gab = municipio[municipio["rede_padronizada"] == rede_gabarito]
    c = g.merge(gab[["ano", "id_municipio", "taxa_alfabetizacao"]], on=["ano", "id_municipio"], how="inner")
    linhas = []
    for col in ["simples", "ponderada"]:
        d = (c[col] - c["taxa_alfabetizacao"]).abs()
        linhas.append({"rede": rede_gabarito, "taxa": col, "municipios": len(c),
                       "diff_mediana": round(d.median(), 4), "diff_p95": round(d.quantile(0.95), 4),
                       "dentro_de_0.05pp": f"{100 * (d <= 0.05).mean():.1f}%"})
    return c, pd.DataFrame(linhas)


resultados = []
for rede, filtro in [("municipal", com_nota["rede_nome"] == "municipal"),
                     ("estadual", com_nota["rede_nome"] == "estadual"),
                     ("publica", com_nota["rede_nome"].isin(["municipal", "estadual"])),
                     ("total", com_nota["rede_nome"].notna())]:
    comparacao, resumo = compara_com_gabarito(com_nota[filtro], rede)
    resultados.append(resumo)
    if rede == "publica":
        comparacao_publica = comparacao

pd.concat(resultados, ignore_index=True)

,rede,taxa,municipios,diff_mediana,diff_p95,dentro_de_0.05pp
0,municipal,simples,10269,0.2707,2.2855,23.3%
1,municipal,ponderada,10269,0.0037,0.0429,95.9%
2,estadual,simples,2139,0.1626,3.6361,41.1%
3,estadual,ponderada,2139,0.0033,0.0692,93.5%
4,publica,simples,10387,0.2810,2.3526,22.3%
5,publica,ponderada,10387,0.0040,0.0497,95.1%
6,total,simples,398,0.0025,0.0048,99.0%
7,total,ponderada,398,0.0025,0.0048,99.0%


A ponderada ganha disparado: nas redes municipal, estadual e pública, ela fecha com o gabarito em ~95% dos municípios dentro de 0,05pp, enquanto a simples só acerta ~20-40%. **Decisão 1 resolvida: a taxa oficial é a média ponderada pelo `peso_aluno`** — e a Gold calcula assim.

E o caso da rede `total` explica o que parecia estranho: ela só existe para 398 municípios, e neles simples e ponderada coincidem. São os municípios onde o peso é constante (todo aluno avaliado, sem correção amostral) — por isso lá o empate.

## Decisão 4 — qual denominador?

Três candidatos: presentes com nota, todos os presentes, todos os avaliados. Testo os três contra o gabarito da rede pública.

In [3]:
pub = alunos[alunos["rede_nome"].isin(["municipal", "estadual"])].copy()
pub["alfa"] = (pub["proficiencia"] >= 743).astype(float)
pub["alfa_peso"] = pub["alfa"] * pub["peso_aluno"]


def taxa_ponderada(sub, nome):
    g = sub.groupby(["ano", "id_municipio"]).agg(
        soma_alfa_peso=("alfa_peso", "sum"), soma_peso=("peso_aluno", "sum")).reset_index()
    g[nome] = 100 * g["soma_alfa_peso"] / g["soma_peso"]
    return g[["ano", "id_municipio", nome]]


gab_publica = municipio[municipio["rede_padronizada"] == "publica"]
teste = (taxa_ponderada(pub[pub["presente"] & ~pub["sem_nota"]], "den_com_nota")
         .merge(taxa_ponderada(pub[pub["presente"]], "den_presentes"), on=["ano", "id_municipio"])
         .merge(taxa_ponderada(pub, "den_todos"), on=["ano", "id_municipio"])
         .merge(gab_publica[["ano", "id_municipio", "taxa_alfabetizacao"]], on=["ano", "id_municipio"], how="inner"))

for col in ["den_com_nota", "den_presentes", "den_todos"]:
    d = (teste[col] - teste["taxa_alfabetizacao"]).abs()
    print(f"{col:14s} mediana={d.median():.4f}  p95={d.quantile(0.95):.4f}  dentro de 0,05pp={100 * (d <= 0.05).mean():.1f}%")

den_com_nota   mediana=0.0040  p95=0.0497  dentro de 0,05pp=95.1%
den_presentes  mediana=0.0040  p95=0.0497  dentro de 0,05pp=95.1%
den_todos      mediana=0.0040  p95=0.0497  dentro de 0,05pp=95.1%


O dado decidiu sozinho: **as três variantes dão idêntico**, porque o `peso_aluno` só existe para presentes com nota (a Silver já tinha mostrado que o nulo do peso acompanha o da proficiência). O denominador efetivo é *presentes com nota* — e a taxa de participação entra na Gold como coluna própria, calculada das contagens, sem sumir com a informação de ausência.

### Os outliers que sobram

Antes de fechar, quero olhar quem não bate: municípios com diferença acima de 1pp mesmo na ponderada.

In [4]:
comparacao_publica["diff"] = (comparacao_publica["ponderada"] - comparacao_publica["taxa_alfabetizacao"]).abs()
outliers = comparacao_publica[comparacao_publica["diff"] > 1]
print(f"outliers (> 1pp): {len(outliers)} de {len(comparacao_publica):,} ({100 * len(outliers) / len(comparacao_publica):.2f}%)")
print(outliers.groupby("ano").size().rename("municipios").to_string())
outliers.nlargest(5, "diff")[["ano", "id_municipio", "ponderada", "taxa_alfabetizacao", "n", "diff"]].round(3)

outliers (> 1pp): 45 de 10,387 (0.43%)
ano
2023    32
2024    13


,ano,id_municipio,ponderada,taxa_alfabetizacao,n,diff
922,2023,2305100,55.609,91.67,71,36.061
2932,2023,3162450,45.920,64.64,75,18.720
106,2023,1304005,57.121,40.10,121,17.021
919,2023,2304905,83.111,99.15,118,16.039
10045,2024,5103908,51.602,66.10,64,14.498


São 45 municípios (0,43%), quase todos de 2023 — o ano da Pesquisa Alfabetiza Brasil, de desenho amostral diferente. Não achei na fonte uma explicação fechada, então a decisão é pragmática: a Gold segue com o cálculo validado (que fecha com mediana de 0,004pp), o check de consistência contra o gabarito entra com tolerância, e os outliers ficam documentados no dicionário de dados em vez de travar a esteira. Se a fonte revisar, o check acusa.

## Decisão 5 — qual snapshot de meta?

As metas têm linhas por ano de referência e o planejamento registrou um caso de revisão (a meta 2024 do Brasil muda entre snapshots). Preciso saber o tamanho disso em cada nível.

In [5]:
metas_municipio = metas[metas["nivel"] == "municipio"]
piv_mun = metas_municipio.pivot_table(index="id_municipio", columns="ano",
                                      values="meta_alfabetizacao_2024", aggfunc="first").dropna()
iguais = int((piv_mun[2023] == piv_mun[2024]).sum())
print(f"municipios nos dois snapshots: {len(piv_mun):,} | meta_2024 igual: {iguais:,} | revisada: {len(piv_mun) - iguais}")

metas_uf = metas[metas["nivel"] == "uf"]
piv_uf = metas_uf.pivot_table(index="sigla_uf", columns="ano",
                              values="meta_alfabetizacao_2024", aggfunc="first")
revisadas = piv_uf[piv_uf.nunique(axis=1) > 1]
print(f"\nUFs com meta_2024 revisada entre snapshots: {len(revisadas)} de {len(piv_uf)}")
revisadas.head(8)

municipios nos dois snapshots: 5,232 | meta_2024 igual: 5,232 | revisada: 0

UFs com meta_2024 revisada entre snapshots: 22 de 24


ano,2023,2024,2025
sigla_uf,,,
AL,49.7,49.7,50.0
AM,56.8,56.8,57.0
AP,47.6,47.6,48.0
BA,43.4,43.4,43.0
ES,69.9,69.9,70.0
GO,68.9,68.9,69.0
MA,60.3,60.3,60.0
MG,63.2,63.2,63.0


O quadro ficou claro: **no nível municipal não houve revisão nenhuma** entre os snapshots 2023 e 2024. As revisões só aparecem no snapshot de 2025, nos níveis Brasil/UF — e são arredondamentos (49,7 → 50,0; 56,8 → 57,0). Como os resultados que a Gold confronta são de 2023 e 2024, a regra fica simples e sem ambiguidade:

**Decisão 5: a meta usada é a vigente no ano do resultado (linha de metas com o mesmo `ano`).** O snapshot de 2025 fica registrado, mas só entra em cena quando houver resultado de 2025 para confrontar.

## Contrato que vai para o script

O que `src/03_gold/metricas_gold.py` implementa, validado aqui:

1. **Taxa de alfabetização = média ponderada pelo `peso_aluno`** dos presentes com nota, com a regra explícita `proficiencia >= 743` (mediana de 0,004pp contra o gabarito; ~95% dos municípios dentro de 0,05pp).
2. **Denominador: presentes com nota** (o peso só existe para eles); taxa de participação como coluna própria.
3. **Meta vigente no ano do resultado** no confronto meta × resultado; 2023 sem meta pactuada (colunas começam em 2024) fica com gap nulo.
4. **Check de consistência** taxa recalculada × gabarito com tolerância, e os 45 outliers de 2023 documentados no dicionário de dados.

Os números deste notebook são os valores de referência para validar a primeira execução do script.